# Native quantum gradient estimation

Keep the original signed three-bit coordinate registers, three-bit fractional auxiliary, all 17 m values, all nine l values, and the two-dimensional example. Classical polynomial coefficients are bound before synthesis. Fixed source oracle callbacks become visible native qfunc composition; no generic QCallable API is claimed. Source helper globals become an explicit immutable Model. Every program has an independent full complex Circuit comparison without phase alignment.


In [ ]:
import math
from pathlib import Path
from typing import Annotated

from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.algorithms.quantum_primitives.gradient_estimation.overview.plots import (
    gradient_plot,
    phase_plot,
    sweep_plot,
)
from examples.synthesis.algorithms.quantum_primitives.gradient_estimation.overview.workflow import (
    L_VALUES,
    M_VALUES,
    Model,
    StateCheck,
    Success,
    phase_table,
    sample,
    success_rate,
    verify_auxiliary,
    verify_program,
    verify_two_dimensional,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    adjoint,
    allocate,
    apply,
    drop,
    hadamard_transform,
    inplace_prepare_amplitudes,
    phase,
    qft,
    qfunc,
    within,
)

output_directory = Path("artifacts/development/gradient_estimation")
checks: list[StateCheck] = []
shots: list[int] = []
phase_tables = []
figures: list[Path] = []
success_results: list[Success] = []


def show_figure(path: Path) -> None:
    figures.append(path)
    display(Image(filename=str(path)))


## Actual auxiliary-register phase kickback

The source starts from the all-one basis state and Fourier-transforms the auxiliary, adds the normalized function modulo its three-bit fractional range, then reverses the preparation. Native basis preparation uses an amplitude vector. The fixed addition is expressed by a separate Fourier within/apply region inside the action; omitting that region would leave a dirty auxiliary. This preserves the physical auxiliary experiment rather than replacing it with direct phase. Direct fractional += and fractional bit views are not claimed.


In [ ]:
@qfunc
def auxiliary_phase(x: Output[Annotated[QNum, (3, True, 0)]]) -> None:
    allocate(x)
    hadamard_transform(x)
    ancilla: Annotated[QNum, (3, True, 3)] = QNum()
    allocate(ancilla)
    with within():
        inplace_prepare_amplitudes([0, 0, 0, 0, 0, 0, 0, 1], 0, ancilla)
        qft(ancilla)
    with apply():
        with within():
            qft(ancilla)
        with apply():
            phase((0.25 * x + 2) * ancilla, 16 * math.pi)
    drop(ancilla)


model = Model(0, 0.5, 0.25)
auxiliary_program = synth.synthesize(
    auxiliary_phase, max_workspace=0, memory_limit_bytes=64 * 1024**2
)
checks.append(verify_auxiliary(auxiliary_program))
print(auxiliary_program.report["resources"])
print(
    "Full complex state, including auxiliary:", fq.run(auxiliary_program).statevector()
)


## Unwrap the measured phase

Sort signed x, round measured turns to five decimals, unwrap with period one, and subtract the x=0 row from both theoretical and measured phases, as in the original helper. Absolute phase is checked separately before this plotting convention.


In [ ]:
rows = phase_table(auxiliary_program, model)
phase_tables.append(rows)
print(rows)
show_figure(phase_plot(model, rows, output_directory, "auxiliary_phase"))


## Native direct phase oracle

For f(u)=a*u*u+b*u+c, with u=l*x/8, the normalized polynomial is a*l*x*x/(8*m)+b*x/m+8*c/(l*m). Python computes those classical coefficients; qfunc constructs the coherent polynomial phase. The auxiliary and direct experiments remain separate.


In [ ]:
@qfunc
def phase_oracle(coefficients: list[float], x: QNum) -> None:
    phase(coefficients[0] * x * x + coefficients[1] * x + coefficients[2], 2 * math.pi)


@qfunc
def phase_only(
    coefficients: list[float], x: Output[Annotated[QNum, (3, True, 0)]]
) -> None:
    allocate(x)
    hadamard_transform(x)
    phase_oracle(coefficients, x)


def build_phase(model: Model) -> synth.Program:
    return synth.synthesize(
        phase_only,
        classical_inputs={"coefficients": list(model.coefficients)},
        max_workspace=0,
        memory_limit_bytes=64 * 1024**2,
    )


model = Model(0, 0.5, 0.25)
linear_phase = build_phase(model)
checks.append(verify_program(linear_phase, model, inverse_fourier=False))
print(linear_phase.report["resources"])
rows = phase_table(linear_phase, model)
phase_tables.append(rows)
print(rows)
show_figure(phase_plot(model, rows, output_directory, "linear_phase"))


In [ ]:
model = Model(0.5, 0.25, 0.1, window=2)
large_phase = build_phase(model)
checks.append(verify_program(large_phase, model, inverse_fourier=False))
print(large_phase.report["resources"])
rows = phase_table(large_phase, model)
phase_tables.append(rows)
print(rows)
show_figure(phase_plot(model, rows, output_directory, "quadratic_phase_large"))


In [ ]:
model = Model(0.5, 0.25, 0.1, window=0.2)
small_phase = build_phase(model)
checks.append(verify_program(small_phase, model, inverse_fourier=False))
print(small_phase.report["resources"])
rows = phase_table(small_phase, model)
phase_tables.append(rows)
print(rows)
show_figure(phase_plot(model, rows, output_directory, "quadratic_phase_small"))


## Gradient construction and actual sampling

Compose Hadamards, the same polynomial phase helper, and inverse QFT. Two-dimensional coordinates are named signed x/y ports rather than a QArray of QNum; their Cartesian domain and bit order are preserved. Each source sample is made explicit as 1000 actual local draws. Success uses the original strict inequality and default tolerance m/16. There is no postselection or injected outcome.


In [ ]:
@qfunc
def gradient_1d(
    coefficients: list[float], x: Output[Annotated[QNum, (3, True, 0)]]
) -> None:
    allocate(x)
    hadamard_transform(x)
    phase_oracle(coefficients, x)
    with adjoint():
        qft(x)


@qfunc
def gradient_2d(
    coefficients: list[float],
    x: Output[Annotated[QNum, (3, True, 0)]],
    y: Output[Annotated[QNum, (3, True, 0)]],
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    phase(
        coefficients[0] * x * x
        + coefficients[1] * y * y
        + coefficients[2] * x * y
        + coefficients[3] * x
        + coefficients[4] * y
        + coefficients[5],
        2 * math.pi,
    )
    with adjoint():
        qft(x)
        qft(y)


def build_gradient(model: Model) -> synth.Program:
    return synth.synthesize(
        gradient_1d,
        classical_inputs={"coefficients": list(model.coefficients)},
        max_workspace=0,
        memory_limit_bytes=64 * 1024**2,
    )


def run_gradient(model: Model, label: str, seed: int) -> Success:
    program = build_gradient(model)
    checks.append(verify_program(program, model, inverse_fourier=True))
    observations = sample(program, shots=1000, seed=seed)
    outcome = success_rate(observations, model, (model.linear,))
    shots.append(outcome.shots)
    success_results.append(outcome)
    print(label, program.report["resources"])
    print(observations)
    print(outcome)
    print(
        "Success bar:",
        "#" * round(50 * outcome.rate) + "-" * (50 - round(50 * outcome.rate)),
    )
    show_figure(gradient_plot(model, observations, output_directory, label))
    return outcome


In [ ]:
negative_linear = run_gradient(Model(0, -0.5, 0.25), "negative_linear", 3000)


In [ ]:
noninteger_linear = run_gradient(Model(0, 0.55, 0.25), "noninteger_linear", 3001)


In [ ]:
quadratic_small = run_gradient(
    Model(0.6, 0.25, 0.1, window=0.1), "quadratic_small", 3002
)
assert quadratic_small.rate > 0.9


In [ ]:
quadratic_large = run_gradient(Model(0.6, 0.25, 0.1, window=1), "quadratic_large", 3003)
assert quadratic_large.rate < 0.5


## Complete m sweep

Retain every original m value and the source tolerance override 0.2. Each point is independently synthesized, executed, compared as a full complex state, and sampled. The green interval is the source theoretical annotation, not a fitted performance guarantee.


In [ ]:
m_rates: list[float] = []
for index, value in enumerate(M_VALUES):
    model = Model(0.6, 0.25, 0.1, window=0.1, m=value)
    program = build_gradient(model)
    checks.append(verify_program(program, model, inverse_fourier=True))
    observations = sample(program, shots=1000, seed=3100 + index)
    outcome = success_rate(observations, model, (0.25,), tolerance=0.2)
    shots.append(outcome.shots)
    m_rates.append(outcome.rate)
    print("m:", value, outcome)
show_figure(sweep_plot(M_VALUES, tuple(m_rates), "m", output_directory))


## Complete l sweep

Retain all nine source l values, m=1 and the source default tolerance. The effect of leaving the locally linear regime is shown by the actual measured rates.


In [ ]:
l_rates: list[float] = []
for index, value in enumerate(L_VALUES):
    model = Model(0.6, 0.25, 0.1, window=value, m=1)
    program = build_gradient(model)
    checks.append(verify_program(program, model, inverse_fourier=True))
    observations = sample(program, shots=1000, seed=3200 + index)
    outcome = success_rate(observations, model, (0.25,))
    shots.append(outcome.shots)
    l_rates.append(outcome.rate)
    print("l:", value, outcome)
show_figure(sweep_plot(L_VALUES, tuple(l_rates), "l", output_directory))


## Two-dimensional gradient

Keep f(u,v)=0.6*u*u-0.4*v*v+0.25*u*v+0.25*u-0.5*v+0.1, l=0.1, m=2 and two signed three-bit coordinates. Both derivative components must pass the same strict tolerance in a single observation.


In [ ]:
window, m = 0.1, 2.0
coefficients = (
    0.6 * window / (8 * m),
    -0.4 * window / (8 * m),
    0.25 * window / (8 * m),
    0.25 / m,
    -0.5 / m,
    0.1 * 8 / (window * m),
)
program_2d = synth.synthesize(
    gradient_2d,
    classical_inputs={"coefficients": list(coefficients)},
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
checks.append(verify_two_dimensional(program_2d))
observations_2d = sample(program_2d, shots=1000, seed=3400)
result_2d = success_rate(
    observations_2d, Model(0.6, 0.25, 0.1, window=0.1), (0.25, -0.5)
)
shots.append(result_2d.shots)
print(program_2d.report["resources"])
print(observations_2d)
print(result_2d)
assert result_2d.rate > 0.9


All 35 source programs preserve their original model coefficients and register sizes. Four complete eight-row phase tables,31actual1000draw batches,ten generated plots and both parameter sweeps are retained. Reports describe synthesized instructions before Runtime basis conversion. No compiler/public API/dependency change,source importer,hardware execution,speedup,scalability or optimality claim.
